# Region Analysis — CODEX cHL

Unsupervised tissue region discovery using the trained CIM backbone.

**Pipeline:**
1. Load the trained CIM/VICReg backbone
2. Precompute sliding-window grid coordinates from each tissue image (shape only, no pixel I/O)
3. Stream patches one-by-one from HDF5, embed in mini-batches → no full-slide load
4. Cluster embeddings with k-means (after PCA)
5. Interpret clusters by their cell-type composition
6. Visualise: UMAP · spatial tissue map · composition bar chart

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────

WORK_DIR      = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/MCA/z_RUNS/CODEX_cHL_CIM_VICReg'
H5_PATH       = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/CODEX_cHL/CODEX_cHL.h5'
MARKERS_PATH  = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/CODEX_cHL/used_markers.txt'

REGION_PATCH_SIZE = 64    # spatial size of each region patch in pixels
                          # 64px ≈ 2× single-cell patch; covers ~4–16 cells per patch
                          # increase to 128 for coarser, more contextual regions
STRIDE            = 32    # step between patches (50 % overlap)

N_CLUSTERS        = 8     # number of unsupervised region types to discover
PCA_COMPONENTS    = 64    # PCA dimensionality before clustering
BATCH_SIZE        = 128   # patches per GPU forward pass

UMAP_MAX_SAMPLES  = 30_000  # subsample for UMAP speed; None = all

SAVE_DIR = '../z_RUNS/region_analysis'

# ───────────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import numpy as np
import h5py
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
import umap
import json

from MCA.src.utils import load_checkpoint

SAVE_DIR = Path(SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Output: {SAVE_DIR.resolve()}')

## 1 · Load the trained backbone

In [ ]:
result   = load_checkpoint(WORK_DIR, device=DEVICE)
model    = result['model']
backbone = model.backbone   # WideModel / CIM encoder
backbone.eval()
print('Backbone loaded.')

# Sanity-check: variable-size forward pass
with torch.no_grad():
    dummy = torch.zeros(1, 41, REGION_PATCH_SIZE, REGION_PATCH_SIZE, device=DEVICE)
    out   = backbone(dummy)
    feat  = out[0].squeeze(-1).squeeze(-1)
    FEAT_DIM = feat.shape[1]
    print(f'Feature dim for {REGION_PATCH_SIZE}×{REGION_PATCH_SIZE} patch: {FEAT_DIM}')

## 2 · Load HDF5 metadata (no pixel data yet)

In [ ]:
def decode(arr):
    return np.array([x.decode() if isinstance(x, bytes) else x for x in arr])

with h5py.File(H5_PATH, 'r') as f:
    all_marker_names = decode(f['marker_names'][:])
    cell_sample_ids  = decode(f['coords']['sample_id'][:])
    cell_dim1        = f['coords']['DIM1'][:].astype(int)   # row  (y)
    cell_dim2        = f['coords']['DIM2'][:].astype(int)   # col  (x)
    cell_annotations = decode(f['annotation'][:])
    unique_samples   = decode(f['sample_ids'][:])

with open(MARKERS_PATH) as fh:
    used_names = np.array([l.strip() for l in fh if l.strip()])

marker2idx     = {m: i for i, m in enumerate(all_marker_names)}
marker_indices = np.array([marker2idx[m] for m in used_names])
N_MARKERS      = len(marker_indices)

print(f'Markers in panel : {len(all_marker_names)}  |  Used: {N_MARKERS}')
print(f'Cells in dataset : {len(cell_dim1):,}')
print(f'Samples ({len(unique_samples)}): {list(unique_samples)}')

## 3 · Precompute sliding-window grid coordinates

Only reads image **shape** from HDF5 — no pixel data loaded yet.

In [ ]:
ps = REGION_PATCH_SIZE

all_meta = []   # list of (sample_id, y, x, img_H, img_W)

with h5py.File(H5_PATH, 'r') as f:
    for sid in unique_samples:
        H, W = f['data'][sid]['image'].shape[:2]   # shape read — no data load
        for y in range(0, H - ps + 1, STRIDE):
            for x in range(0, W - ps + 1, STRIDE):
                all_meta.append((sid, y, x, H, W))

print(f'Total region patches : {len(all_meta):,}')
for sid in unique_samples:
    n = sum(1 for m in all_meta if m[0] == sid)
    print(f'  {sid}: {n:,} patches')

## 4 · Stream patches from HDF5 and embed

Patches are read one-by-one from HDF5 (lazy slice, no full-slide load), 
batched in RAM, and flushed to the GPU every `BATCH_SIZE` patches.

In [ ]:
@torch.no_grad()
def flush_batch(batch_list, backbone, device):
    arr   = np.array(batch_list, dtype=np.float32)      # (B, C, ps, ps)
    t     = torch.from_numpy(arr).to(device)
    feats = backbone(t)
    if isinstance(feats, (tuple, list)):
        feats = feats[0]
    feats = feats.squeeze(-1).squeeze(-1)               # (B, D)
    return F.normalize(feats, dim=1).cpu().numpy()


embeddings    = []
batch_patches = []

with h5py.File(H5_PATH, 'r') as f:
    for sid, y, x, H, W in tqdm(all_meta, desc='Loading & embedding patches',
                                 unit='patch', dynamic_ncols=True):
        # Read only this patch from HDF5  (lazy slice — no full image load)
        patch = f['data'][sid]['image'][y : y + ps, x : x + ps, :].astype(np.float32)
        patch = patch[:, :, marker_indices]             # (ps, ps, n_used)
        batch_patches.append(patch.transpose(2, 0, 1)) # (n_used, ps, ps)

        if len(batch_patches) == BATCH_SIZE:
            embeddings.append(flush_batch(batch_patches, backbone, DEVICE))
            batch_patches = []

    if batch_patches:   # flush tail
        embeddings.append(flush_batch(batch_patches, backbone, DEVICE))

embeddings = np.concatenate(embeddings, axis=0)
print(f'\nEmbeddings: {embeddings.shape}   (n_patches × feature_dim)')

## 5 · PCA + k-means clustering

In [ ]:
pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
emb_pca = pca.fit_transform(embeddings)
print(f'PCA explained variance ({PCA_COMPONENTS} components): {pca.explained_variance_ratio_.sum():.1%}')

kmeans = MiniBatchKMeans(
    n_clusters  = N_CLUSTERS,
    random_state= 42,
    n_init      = 10,
    batch_size  = min(4096, len(emb_pca)),
    max_iter    = 300,
)
cluster_labels = kmeans.fit_predict(emb_pca)
print(f'\nCluster sizes:')
for c, n in enumerate(np.bincount(cluster_labels)):
    print(f'  Cluster {c}: {n:,} patches ({n/len(cluster_labels):.1%})')

In [ ]:
# ── Optional: sweep over k to find elbow ───────────────────────────────────
from sklearn.metrics import silhouette_score

ks      = [4, 6, 8, 10, 12, 16]
inertia = []
sil     = []

idx_sub = np.random.choice(len(emb_pca), min(10_000, len(emb_pca)), replace=False)
X_sub   = emb_pca[idx_sub]

for k in tqdm(ks, desc='k sweep'):
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5, batch_size=2048)
    km.fit(X_sub)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_sub, km.labels_, metric='euclidean', sample_size=5000))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(ks, inertia, 'o-'); axes[0].set(xlabel='k', ylabel='Inertia', title='Elbow')
axes[1].plot(ks, sil,     'o-'); axes[1].set(xlabel='k', ylabel='Silhouette', title='Silhouette')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'k_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best silhouette: k={ks[int(np.argmax(sil))]}  ({max(sil):.3f})')

## 6 · UMAP of region embeddings

In [ ]:
n_umap = min(UMAP_MAX_SAMPLES, len(embeddings)) if UMAP_MAX_SAMPLES else len(embeddings)
idx_u  = np.random.choice(len(embeddings), n_umap, replace=False)

reducer = umap.UMAP(
    n_components = 2,
    n_neighbors  = 30,
    min_dist     = 0.1,
    metric       = 'cosine',
    random_state = 42,
    verbose      = True,
)
umap_emb = reducer.fit_transform(embeddings[idx_u])

cmap = plt.cm.get_cmap('tab10', N_CLUSTERS)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    umap_emb[:, 0], umap_emb[:, 1],
    c=cluster_labels[idx_u], cmap=cmap,
    vmin=-0.5, vmax=N_CLUSTERS - 0.5, s=3, alpha=0.6,
)
plt.colorbar(sc, ax=ax, label='Cluster', ticks=range(N_CLUSTERS))
ax.set(title=f'UMAP of region embeddings  (k={N_CLUSTERS}, n={n_umap:,})',
       xlabel='UMAP 1', ylabel='UMAP 2')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'umap_region_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Spatial tissue maps

In [ ]:
cmap      = plt.cm.get_cmap('tab10', N_CLUSTERS)
n_samples = len(unique_samples)
fig, axes = plt.subplots(1, n_samples, figsize=(7 * n_samples, 6), squeeze=False)

for ax, sid in zip(axes[0], unique_samples):
    sample_entries = [(i, y, x, H, W) for i, (s, y, x, H, W) in enumerate(all_meta) if s == sid]
    if not sample_entries:
        ax.set_visible(False); continue

    _, _, _, img_H, img_W = sample_entries[0]
    canvas = np.full((img_H, img_W, 4), fill_value=[0.85, 0.85, 0.85, 1.0])

    for i, y, x, _, _ in sample_entries:
        canvas[y : y + ps, x : x + ps] = np.array(cmap(int(cluster_labels[i])))

    ax.imshow(canvas, aspect='equal')
    ax.set_title(sid, fontsize=9)
    ax.axis('off')

legend_patches = [mpatches.Patch(color=cmap(c), label=f'Cluster {c}') for c in range(N_CLUSTERS)]
fig.legend(handles=legend_patches, loc='lower center', ncol=N_CLUSTERS, frameon=False, fontsize=9)
fig.suptitle(f'Spatial region map  (patch={ps}px, stride={STRIDE}px)', fontsize=12)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(SAVE_DIR / 'spatial_region_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Per-cluster cell-type composition

For each region patch, find all cells whose centre `(DIM1, DIM2)` falls inside it,
then aggregate annotations to get the cell-type distribution per cluster.

In [ ]:
# Build per-sample cell index for fast lookup
sample_to_cell_idx = defaultdict(list)
for idx, sid in enumerate(cell_sample_ids):
    sample_to_cell_idx[sid].append(idx)

cluster_cells = defaultdict(list)   # cluster_id -> [cell_type, ...]

for i, (sid, y, x, H, W) in enumerate(tqdm(all_meta, desc='Assigning cells to regions',
                                            unit='patch', dynamic_ncols=True)):
    cl = int(cluster_labels[i])
    for cidx in sample_to_cell_idx[sid]:
        cy, cx = int(cell_dim1[cidx]), int(cell_dim2[cidx])
        if y <= cy < y + ps and x <= cx < x + ps:
            cluster_cells[cl].append(cell_annotations[cidx])

all_types = sorted({a for cells in cluster_cells.values() for a in cells})
print(f'Cell types found: {all_types}')

In [ ]:
composition = {}
for cl in range(N_CLUSTERS):
    cells = cluster_cells[cl]
    total = len(cells)
    counts = Counter(cells)
    composition[cl] = {ct: (counts.get(ct, 0) / total if total else 0.0) for ct in all_types}

comp_df = pd.DataFrame(composition).T[all_types]

fig, ax = plt.subplots(figsize=(14, 5))
comp_df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', legend=True, width=0.8)
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Cell-type fraction', fontsize=11)
ax.set_title(f'Cell-type composition per region cluster  (k={N_CLUSTERS})', fontsize=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
ax.set_xticklabels([f'C{c}' for c in range(N_CLUSTERS)], rotation=0)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'cluster_composition.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Dominant cell type per cluster ===')
print(f'{"Cluster":<12} {"Dominant type":<22} {"Fraction":>8}  {"N cells":>9}  {"N patches":>10}')
print('-' * 68)
for cl in range(N_CLUSTERS):
    dominant = max(composition[cl], key=composition[cl].get)
    frac     = composition[cl][dominant]
    n_cells  = len(cluster_cells[cl])
    n_pat    = int(np.sum(cluster_labels == cl))
    print(f'  C{cl:<10} {dominant:<22} {frac:>7.1%}  {n_cells:>9,}  {n_pat:>10,}')

## 9 · Per-sample spatial map with composition pie

In [ ]:
n_rows = len(unique_samples)
cmap   = plt.cm.get_cmap('tab10', N_CLUSTERS)

fig, axes = plt.subplots(n_rows, 2, figsize=(14, 6 * n_rows),
                         gridspec_kw={'width_ratios': [3, 1]})
if n_rows == 1:
    axes = axes[np.newaxis, :]

for row, sid in enumerate(unique_samples):
    ax_map, ax_pie = axes[row, 0], axes[row, 1]

    sample_entries = [(i, y, x, H, W) for i, (s, y, x, H, W) in enumerate(all_meta) if s == sid]
    if not sample_entries:
        ax_map.set_visible(False); ax_pie.set_visible(False); continue

    _, _, _, img_H, img_W = sample_entries[0]
    canvas = np.full((img_H, img_W, 4), fill_value=[0.9, 0.9, 0.9, 1.0])
    sample_cluster_counts = Counter()

    for i, y, x, _, _ in sample_entries:
        cl = int(cluster_labels[i])
        canvas[y : y + ps, x : x + ps] = np.array(cmap(cl))
        sample_cluster_counts[cl] += 1

    ax_map.imshow(canvas, aspect='equal')
    ax_map.set_title(sid, fontsize=10)
    ax_map.axis('off')

    pie_cls    = sorted(sample_cluster_counts)
    ax_pie.pie(
        [sample_cluster_counts[c] for c in pie_cls],
        labels=[f'C{c}' for c in pie_cls],
        colors=[cmap(c) for c in pie_cls],
        autopct='%1.0f%%', startangle=90, textprops={'fontsize': 8},
    )
    ax_pie.set_title('Cluster area\nfraction', fontsize=9)

legend_patches = [mpatches.Patch(color=cmap(c), label=f'C{c}') for c in range(N_CLUSTERS)]
fig.legend(handles=legend_patches, loc='lower center', ncol=N_CLUSTERS, frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(SAVE_DIR / 'spatial_map_with_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Save results

In [ ]:
results = {
    'settings': {
        'region_patch_size': REGION_PATCH_SIZE,
        'stride'           : STRIDE,
        'n_clusters'       : N_CLUSTERS,
        'pca_components'   : PCA_COMPONENTS,
    },
    'stats': {
        'total_patches'    : int(len(all_meta)),
        'feature_dim'      : int(FEAT_DIM),
        'pca_explained_var': float(pca.explained_variance_ratio_.sum()),
        'kmeans_inertia'   : float(kmeans.inertia_),
    },
    'cluster_sizes'       : {int(c): int(n) for c, n in enumerate(np.bincount(cluster_labels))},
    'cluster_composition' : {
        int(cl): {ct: float(v) for ct, v in comp_df.loc[cl].items()}
        for cl in range(N_CLUSTERS)
    },
}

with open(SAVE_DIR / 'region_results.json', 'w') as f:
    json.dump(results, f, indent=2)

meta_df = pd.DataFrame(
    [(sid, y, x, H, W, int(cluster_labels[i]))
     for i, (sid, y, x, H, W) in enumerate(all_meta)],
    columns=['sample_id', 'y', 'x', 'img_H', 'img_W', 'cluster']
)
meta_df.to_csv(SAVE_DIR / 'region_assignments.csv', index=False)

print(f'Saved to {SAVE_DIR.resolve()}/')
for fname in ['region_results.json', 'region_assignments.csv',
              'umap_region_clusters.png', 'spatial_region_map.png',
              'spatial_map_with_pie.png', 'cluster_composition.png', 'k_sweep.png']:
    print(f'  {fname}')